<span style="color:lightgreen">

# Notebook extra 2.2

En este notebook se experimenta con los parámetros de transcripción con traducción de whisper en cascada.

En el ejercicio anterior (extra 2.1) hemos probado diferentes parámetros para la transcripción con whisper, ahora usaremos el mejor resultado de ese experimento para su traducción.

En lugar de usar NLLB, usaremos el modelo `google/madlad400-3b-mt`

Covost2 pt-en

</span>

# Cascade ST Baseline experiment using Whisper along with NLLB and Europarl-ST (Spanish to English)

In this notebook, we are going to learn how to use Whisper and Meta's large pre-trained models [NLLB](https://huggingface.co/docs/transformers/model_doc/nllb) in a cascade ST approach on the Europarl-ST using the English-Spanish translation pair.

First, we import automatic transcriptions, references and translations (and their clean counterparts) generated by the ASR baseline experiment using Whisper

In [1]:
import os

print(os.getcwd())
if not os.getcwd().endswith("app"):
    os.chdir("../app")
    print(os.getcwd())


%load_ext autoreload
%autoreload 2
%matplotlib inline

from src.config import Configuration

CONFIG = Configuration(
    model_name="google/madlad400-3b-mt",
    trans_code='<2en> ' # to indicate the model that the target language is English
)

/home/turbotowerlnx/Documents/Master/TA/TA-Portuguesse-English-ST/notebooks_pt_en
/home/turbotowerlnx/Documents/Master/TA/TA-Portuguesse-English-ST/app


In [2]:
from datasets import load_dataset

raw_datasets = load_dataset("csv", data_files=os.path.join(CONFIG.RESULTS_PATH, "Extra2.1-L4.1_ASR_Whisper_Baseline_Covost2_pt-en.csv"))

# Filter out rows with null values in hypothesis or translation
raw_datasets = raw_datasets.filter(lambda x: x["hypothesis"] is not None and x["translation"] is not None)

print(raw_datasets)

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0', 'hypothesis', 'reference', 'translation', 'hypothesis_clean', 'reference_clean', 'translation_clean'],
        num_rows: 4021
    })
})


Show the first sample for each dataset feature

In [3]:
print(raw_datasets["train"][:1]["hypothesis"])
print(raw_datasets["train"][:1]["reference"])
print(raw_datasets["train"][:1]["translation"])
print(raw_datasets["train"][:1]["hypothesis_clean"])
print(raw_datasets["train"][:1]["reference_clean"])
print(raw_datasets["train"][:1]["translation_clean"])

[' e dia de ir emprestado as pessoas daudei.']
['Pedir dinheiro emprestado às pessoas da aldeia']
['Borrow money from people in the village']
[' e dia de ir emprestado as pessoas daudei ']
['pedir dinheiro emprestado às pessoas da aldeia']
['borrow money from people in the village']


<p style="page-break-after:always;"></p>

Now we load the pre-trained tokenizer for the NLLB model and apply it to the Spanish-English pair:

In [4]:
from maikol_utils.print_utils import print_color
from transformers import AutoTokenizer

max_tok_length = 275
# checkpoint = "facebook/nllb-200-distilled-600M"
checkpoint = CONFIG.model_name 
print_color(f"The model checkpoint is: {checkpoint}")

# from flores200_codes import flores_codes
# tokenizer = AutoTokenizer.from_pretrained(
#     checkpoint, 
#     padding=True, 
#     pad_to_multiple_of=8, 
#     src_lang=CONFIG.src_code, 
#     tgt_lang=CONFIG.tgt_code, 
#     truncation=False, 
#     max_length=max_tok_length,
#     )
tokenizer = AutoTokenizer.from_pretrained(
    checkpoint,
    padding=True,
    truncation=False,
    max_length=max_tok_length,
)

The model checkpoint is: google/madlad400-3b-mt


We will use the [Dataset.map() function](https://huggingface.co/docs/datasets/en/package_reference/main_classes#datasets.Dataset.map). This allows us some extra flexibility, if we need more preprocessing done than just tokenization. The map() method works by applying a function on each element of the dataset.

In our case, each sample pair is going to be preprocessed according to the training needs of the model that is to be used. In this case we tokenize speech transcriptions and translations to perform ST cascade experiments

In [5]:
def preprocess_function(sample):
    model_inputs = tokenizer(
        sample["hypothesis"], 
        text_target = sample["translation"],
        truncation=True,
        max_length=max_tok_length
        )
    return model_inputs

Now, we can apply the preprocess_function to the raw datasets

In [6]:
tokenized_datasets = raw_datasets.map(preprocess_function, batched=True)

<p style="page-break-after:always;"></p>

bitsandbytes is a quantization library with a Transformers integration. With this integration, you can quantize a model to 8 or 4-bits and enable many other options by configuring the BitsAndBytesConfig class. For example, you can:

<ul>
<li>set load_in_4bit=True to quantize the model to 4-bits when you load it</li>
<li>set bnb_4bit_quant_type="nf4" to use a special 4-bit data type for weights initialized from a normal distribution</li>
<li>set bnb_4bit_use_double_quant=True to use a nested quantization scheme to quantize the already quantized weights</li>
<li>set bnb_4bit_compute_dtype=torch.bfloat16 to use bfloat16 for faster computation</li>
</ul>

In [7]:
import torch
from transformers import BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

Pass the quantization_config to the from_pretrained method.

In [8]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(
    checkpoint,
    quantization_config=quantization_config,
    # device_map="auto"
)

KeyboardInterrupt: 

<p style="page-break-after:always;"></p>

## Inference

At inference time, it is recommended to use the [generate function](https://huggingface.co/docs/transformers/main_classes/text_generation). This method takes care of encoding the input and auto-regressively generates the decoder output. Check out [this blog post](https://huggingface.co/blog/how-to-generate) to know all the details about generating text with Transformers.
There’s also [this blog post](https://huggingface.co/blog/encoder-decoder#encoder-decoder) which explains how generation works in general in encoder-decoder models.

Let us first load the default inference parameters of NLLB.

In [ ]:
from transformers import GenerationConfig

generation_config = GenerationConfig.from_pretrained(
    checkpoint,
)

print(generation_config)

GenerationConfig {
  "decoder_start_token_id": 0,
  "eos_token_id": 2,
  "pad_token_id": 1
}



In [ ]:
test_batch_size = 32
batch_tokenized_test = tokenized_datasets['train'].batch(test_batch_size)

<span style="color:lightgreen">
Definición experimentos
</span>

In [ ]:
experiments = [
    {"name": "Default (Greedy)", "params": {}}, # Uses defaults from generation_config
    {"name": "Beam Search (n=5)", "params": {"num_beams": 5, "early_stopping": True}},
    {"name": "Multinomial Sampling (T=0.7)", "params": {"do_sample": True, "temperature": 0.7}},
    {"name": "Nucleus Sampling (p=0.9)", "params": {"do_sample": True, "top_p": 0.9, "temperature": 1.0}},
    {"name": "Contrastive Search", "params": {"penalty_alpha": 0.6, "top_k": 4}},
]


<p style="page-break-after:always;"></p>

Processing in batches to add padding and convert to tensors, then perform inference with num_beams = 1 and do_sample = False, that is, greedy search.

<span style="color:lightgreen">

Le ponemos al modelo el token con la instrucción para que sepa qué hacer
<span>

In [ ]:
from tqdm import tqdm
from maikol_utils.print_utils import print_color, print_separator
from whisper.normalizers.basic import BasicTextNormalizer
from evaluate import load


all_results = {}
metric = load("sacrebleu")
comet_metric = load('comet')
normalizer = BasicTextNormalizer()
number_of_batches = len(batch_tokenized_test["hypothesis"])

references = tokenizer.batch_decode(tokenized_datasets["train"]["labels"], skip_special_tokens=True)
references_clean = [normalizer(text) for text in references]

for exp in experiments:
    print_separator(exp['name'], sep_type="SUPER")
    output_sequences = []
    
    for i in tqdm(range(number_of_batches)):
        inputs = tokenizer(
            [CONFIG.trans_code + text for text in batch_tokenized_test["hypothesis"][i]], 
            max_length=max_tok_length, 
            truncation=False, 
            return_tensors="pt", 
            padding=True,
        )
        
        # We merge generation_config with experiment params
        # Note: forced_bos_token_id is removed as it's not needed for MADLAD-400
        output_batch = model.generate(
            input_ids=inputs["input_ids"].cuda(), 
            attention_mask=inputs["attention_mask"].cuda(), 
            generation_config=generation_config,
            max_length=max_tok_length, 
            **exp["params"]
        )
        output_sequences.extend(output_batch.cpu())
    

    decoded_preds = tokenizer.batch_decode(output_sequences, skip_special_tokens=True)
    decoded_preds_clean = [normalizer(text) for text in decoded_preds]

    blue_score = metric.compute(predictions=decoded_preds, references=references)
    blue_score_clean = metric.compute(predictions=decoded_preds_clean, references=references_clean)
    print_color(f'BLEU score:       {blue_score["score"]:.2f}', color="cyan")
    print_color(f'BLEU score Clean: {blue_score_clean["score"]:.2f}', color="cyan")

    comet_score = comet_metric.compute(predictions=decoded_preds, references=references, sources=raw_datasets["train"]["hypothesis"])
    comet_score_clean = comet_metric.compute(predictions=decoded_preds_clean, references=references_clean, sources=raw_datasets["train"]["hypothesis"])
    print_color(f"COMET score:       {comet_score['mean_score']:.2%}", color="cyan")
    print_color(f"COMET score Clean: {comet_score_clean['mean_score']:.2%}", color="cyan")


    all_results[exp["name"]] = {}
    all_results[exp["name"]]["output"] = output_sequences
    all_results[exp["name"]]["BLUE"] = blue_score["score"]
    all_results[exp["name"]]["BLUE_CLEAN"] = blue_score_clean["score"]
    all_results[exp["name"]]["COMET"] = comet_score['mean_score']
    all_results[exp["name"]]["COMET_CLEAN"] = comet_score_clean['mean_score']


  0%|          | 0/126 [00:00<?, ?it/s]/home/turbotowerlnx/Documents/Master/TA/TA-Portuguesse-English-ST/venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:2914: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
100%|██████████| 126/126 [01:28<00:00,  1.42it/s]


In [ ]:
# Find best scores
best_bleu = max(all_results.items(), key=lambda x: x[1]["BLUE"])
best_bleu_clean = max(all_results.items(), key=lambda x: x[1]["BLUE_CLEAN"])
best_comet = max(all_results.items(), key=lambda x: x[1]["COMET"])
best_comet_clean = max(all_results.items(), key=lambda x: x[1]["COMET_CLEAN"])

print_separator("BEST SCORES", sep_type="SUPER")

print_color(f"Best BLEU Score (Non-normalized):", color="yellow")
print_color(f" - Experiment: {best_bleu[0]}", color="cyan")
print_color(f" - Score: {best_bleu[1]['BLUE']:.2f}", color="green")

print_color(f"Best BLEU Score (Normalized):", color="yellow")
print_color(f" - Experiment: {best_bleu_clean[0]}", color="cyan")
print_color(f" - Score: {best_bleu_clean[1]['BLUE_CLEAN']:.2f}", color="green")

print_color(f"Best COMET Score (Non-normalized):", color="yellow")
print_color(f" - Experiment: {best_comet[0]}", color="cyan")
print_color(f" - Score: {best_comet[1]['COMET']:.2%}", color="green")

print_color(f"Best COMET Score (Normalized):", color="yellow")
print_color(f" - Experiment: {best_comet_clean[0]}", color="cyan")
print_color(f" - Score: {best_comet_clean[1]['COMET_CLEAN']:.2%}", color="green")

# Show two translation examples from the best overall experiment (using best BLEU clean as criterion)
print_separator("TRANSLATION EXAMPLES", sep_type="SUPER")
print_color(f"\nShowing examples from: {best_bleu_clean[0]}", color="yellow")

best_outputs = best_bleu_clean[1]["output"]
best_translations = tokenizer.batch_decode(best_outputs[:2], skip_special_tokens=True)

for idx, (source, reference, translation) in enumerate(zip(
    raw_datasets["train"]["hypothesis"][:2],
    references[:2],
    best_translations
), 1):
    print_color(f"\nExample {idx}:", color="magenta")
    print_color(f"  Source (PT):      {source}", color="white")
    print_color(f"  Reference (EN):   {reference}", color="white")
    print_color(f"  Translation (EN): {translation}", color="cyan")